# Notebook 2: Huấn luyện Music Prior (GPT)
**Dữ liệu:** Lakh MIDI Clean Dataset
**Output:** music_prior.pt + ảnh đánh giá

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import os, glob, random, math
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family':'serif','font.size':11,'axes.titlesize':12,
    'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.3,'grid.linestyle':'--'})
os.system('pip install pretty_midi -q'); import pretty_midi
SAVE='/kaggle/working/'
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Tải và phân tích Lakh MIDI Dataset

In [ ]:
MIDI_DIR='/kaggle/input/datasets/imsparsh/lakh-midi-clean/'
midi_files=[]
for root,dirs,files in os.walk(MIDI_DIR):
    for f in files:
        if f.lower().endswith(('.mid','.midi')): midi_files.append(os.path.join(root,f))
print(f'Tim thay {len(midi_files)} file MIDI')
random.seed(42); random.shuffle(midi_files)
midi_files=midi_files[:5000]
print(f'Su dung {len(midi_files)} file')


In [ ]:
def midi_to_tokens(path,max_notes=256):
    try:
        pm=pretty_midi.PrettyMIDI(path)
        best=max(pm.instruments,key=lambda x:len(x.notes),default=None)
        if best is None or len(best.notes)<8: return None
        notes=sorted(best.notes,key=lambda n:n.start)[:max_notes]
        tokens=[]; prev_end=0.0
        for note in notes:
            if note.start-prev_end>0.2: tokens.append(128)
            tokens.append(int(np.clip(note.pitch,0,127)))
            prev_end=note.end
        return tokens if len(tokens)>=16 else None
    except: return None

print('Trich xuat tokens...')
all_tokens=[]
pitch_all=[]
for i,f in enumerate(midi_files):
    t=midi_to_tokens(f)
    if t:
        all_tokens.append(t)
        pitch_all.extend([x for x in t if x<128])
    if (i+1)%1000==0: print(f'  {i+1}/{len(midi_files)} | ok={len(all_tokens)}')
print(f'Tong: {len(all_tokens)} sequences hop le')
print(f'Do dai TB: {np.mean([len(t) for t in all_tokens]):.0f} tokens')


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4))
fig.suptitle('Hình T2.1: Phân tích Lakh MIDI Dataset',fontsize=13,fontweight='bold')
# (a) pitch histogram
axes[0].hist(pitch_all,bins=50,color='mediumpurple',edgecolor='none',alpha=0.8)
axes[0].set_title('(a) Phân bố MIDI pitch'); axes[0].set_xlabel('Pitch (0-127)'); axes[0].set_ylabel('Tần số')
axes[0].axvline(np.mean(pitch_all),color='red',ls='--',lw=1.5,label=f'TB={np.mean(pitch_all):.1f}')
axes[0].legend()
# (b) sequence length
lengths=[len(t) for t in all_tokens]
axes[1].hist(lengths,bins=40,color='steelblue',edgecolor='none',alpha=0.8)
axes[1].set_title('(b) Phân bố độ dài sequence'); axes[1].set_xlabel('Số tokens'); axes[1].set_ylabel('Số sequences')
axes[1].axvline(np.mean(lengths),color='red',ls='--',lw=1.5,label=f'TB={np.mean(lengths):.0f}')
axes[1].legend()
# (c) REST ratio
rest_ratios=[t.count(128)/len(t) for t in all_tokens]
axes[2].hist(rest_ratios,bins=30,color='coral',edgecolor='none',alpha=0.8)
axes[2].set_title('(c) Tỉ lệ token REST mỗi bản'); axes[2].set_xlabel('Tỉ lệ REST'); axes[2].set_ylabel('Số sequences')
axes[2].axvline(np.mean(rest_ratios),color='red',ls='--',lw=1.5,label=f'TB={np.mean(rest_ratios):.2f}')
axes[2].legend()
plt.tight_layout(); plt.savefig(SAVE+'prior_fig1_midi_analysis.png',dpi=150,bbox_inches='tight'); plt.show()
print('Saved prior_fig1_midi_analysis.png')


## 2. Dataset và Model

In [ ]:
class MusicPriorDataset(Dataset):
    def __init__(self,seqs,seq_len=128):
        self.data=[]
        for s in seqs:
            for start in range(0,len(s)-seq_len,seq_len//2):
                chunk=s[start:start+seq_len+1]
                if len(chunk)==seq_len+1: self.data.append(chunk)
    def __len__(self): return len(self.data)
    def __getitem__(self,i):
        t=self.data[i]
        return torch.tensor(t[:-1],dtype=torch.long),torch.tensor(t[1:],dtype=torch.long)

random.shuffle(all_tokens); n=int(0.9*len(all_tokens))
train_ds=MusicPriorDataset(all_tokens[:n])
val_ds=MusicPriorDataset(all_tokens[n:])
train_loader=DataLoader(train_ds,batch_size=64,shuffle=True,drop_last=True)
val_loader=DataLoader(val_ds,batch_size=64)
print(f'Train:{len(train_ds)} | Val:{len(val_ds)}')


In [ ]:
class MusicPrior(nn.Module):
    def __init__(self,vocab=130,d_model=128,nhead=4,num_layers=4,max_len=512):
        super().__init__()
        self.embed=nn.Embedding(vocab,d_model,padding_idx=129)
        self.pos_enc=nn.Embedding(max_len,d_model)
        enc=nn.TransformerEncoderLayer(d_model=d_model,nhead=nhead,dim_feedforward=256,dropout=0.1,batch_first=True)
        self.transformer=nn.TransformerEncoder(enc,num_layers=num_layers)
        self.head=nn.Linear(d_model,vocab)
    def forward(self,tokens):
        B,T=tokens.shape
        mask=torch.triu(torch.ones(T,T,device=tokens.device),diagonal=1).bool()
        pos=torch.arange(T,device=tokens.device).unsqueeze(0)
        return self.head(self.transformer(self.embed(tokens)+self.pos_enc(pos),mask=mask))
    def get_hidden(self,tokens):
        B,T=tokens.shape
        mask=torch.triu(torch.ones(T,T,device=tokens.device),diagonal=1).bool()
        pos=torch.arange(T,device=tokens.device).unsqueeze(0)
        return self.transformer(self.embed(tokens)+self.pos_enc(pos),mask=mask)
    @torch.no_grad()
    def generate(self,primer,max_len=64,temperature=1.0):
        self.eval(); tokens=primer.clone()
        for _ in range(max_len-tokens.shape[1]):
            logits=self.forward(tokens)[:,-1,:]/temperature
            tokens=torch.cat([tokens,torch.multinomial(F.softmax(logits,dim=-1),1)],dim=1)
        return tokens

model=MusicPrior().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Random baseline perplexity: {math.exp(math.log(130)):.2f}')


## 3. Huấn luyện

In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=5e-4,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=30)
criterion=nn.CrossEntropyLoss(ignore_index=129)
EPOCHS=30; train_losses,val_losses,perplexities=[],[],[]

for epoch in range(EPOCHS):
    model.train(); tl=0
    for xb,yb in train_loader:
        xb,yb=xb.to(device),yb.to(device)
        optimizer.zero_grad(); loss=criterion(model(xb).reshape(-1,130),yb.reshape(-1))
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step(); tl+=loss.item()
    model.eval(); vl=0
    with torch.no_grad():
        for xb,yb in val_loader:
            xb,yb=xb.to(device),yb.to(device)
            vl+=criterion(model(xb).reshape(-1,130),yb.reshape(-1)).item()
    scheduler.step()
    tl_avg=tl/len(train_loader); vl_avg=vl/len(val_loader)
    train_losses.append(tl_avg); val_losses.append(vl_avg); perplexities.append(math.exp(vl_avg))
    if (epoch+1)%5==0:
        print(f'Epoch {epoch+1:2d}/{EPOCHS} | train={tl_avg:.4f} | val={vl_avg:.4f} | ppl={math.exp(vl_avg):.2f}')
print('Done!')


## 4. Đánh giá và biểu đồ

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4))
fig.suptitle('Hình T2.2: Kết quả huấn luyện Music Prior',fontsize=13,fontweight='bold')

# (a) Loss curve
axes[0].plot(train_losses,color='steelblue',lw=2,label='Train')
axes[0].plot(val_losses,color='coral',lw=2,label='Val')
axes[0].axhline(math.log(130),color='gray',ls=':',lw=1.5,label=f'Random baseline={math.log(130):.2f}')
axes[0].set_title('(a) Loss (CrossEntropy)'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=9)

# (b) Perplexity
axes[1].plot(perplexities,color='mediumpurple',lw=2)
axes[1].axhline(130,color='gray',ls=':',lw=1.5,label='Random=130')
axes[1].set_title('(b) Perplexity trên val set'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Perplexity')
axes[1].legend(fontsize=9)
axes[1].annotate(f'Final: {perplexities[-1]:.2f}',xy=(len(perplexities)-1,perplexities[-1]),
    xytext=(-20,-20),textcoords='offset points',arrowprops=dict(arrowstyle='->',color='red'),color='red',fontsize=9)

# (c) Generated melody comparison
model.eval()
primers_labels=[([60,64,67],'C major (Do-Mi-Sol)'),([57,60,63],'A minor (La-Do-Mi)'),([55,59,62],'G dominant')]
for pi,(primer_notes,label) in enumerate(primers_labels):
    primer=torch.tensor([primer_notes],dtype=torch.long).to(device)
    gen=model.generate(primer,max_len=20,temperature=0.8)[0,3:].cpu().numpy()
    axes[2].plot(range(len(gen)),gen,marker='o',markersize=4,lw=1.5,label=label,alpha=0.8)
axes[2].set_title('(c) Melody sinh ra từ 3 primer khác nhau')
axes[2].set_xlabel('Vị trí nốt'); axes[2].set_ylabel('MIDI pitch')
axes[2].set_ylim(40,90); axes[2].legend(fontsize=9)

plt.tight_layout(); plt.savefig(SAVE+'prior_fig2_training_results.png',dpi=150,bbox_inches='tight'); plt.show()
print('Saved prior_fig2_training_results.png')
print(f'Final perplexity: {perplexities[-1]:.2f} (random={130}, improvement: x{130/perplexities[-1]:.1f})')


## 5. Lưu model

## Ảnh riêng: Note Transition Matrix (Bigram)

In [ ]:
# Xay dung bigram matrix: P(note_j | note_i)
bigram = np.zeros((128, 128), dtype=np.float32)
for seq in all_tokens[:2000]:
    notes = [t for t in seq if t < 128]
    for a, b in zip(notes[:-1], notes[1:]):
        bigram[a, b] += 1
bigram = bigram / (bigram.sum(axis=1, keepdims=True) + 1e-8)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(bigram[48:84, 48:84], cmap='hot_r', aspect='auto', vmin=0, vmax=0.15)
plt.colorbar(im, ax=ax)
ticks = list(range(0, 36, 6)); tick_labels = [str(48+i) for i in ticks]
ax.set_xticks(ticks); ax.set_xticklabels(tick_labels)
ax.set_yticks(ticks); ax.set_yticklabels(tick_labels)
ax.set_title('Hình P-01: Ma trận chuyển tiếp nốt (Bigram)\n'
             'P(nốt j | nốt i) trong phạm vi pitch 48-84', fontsize=12)
ax.set_xlabel('Nốt tiếp theo (j)'); ax.set_ylabel('Nốt hiện tại (i)')
plt.tight_layout()
plt.savefig(SAVE + 'prior_p01_bigram_matrix.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved prior_p01_bigram_matrix.png')


## Ảnh riêng: Temperature Effect on Generation

In [ ]:
model.eval()
primer = torch.tensor([[60, 64, 67]], dtype=torch.long).to(device)
temps = [0.5, 0.8, 1.0, 1.2, 1.5]

fig, axes = plt.subplots(len(temps), 1, figsize=(12, 10), sharex=True)
fig.suptitle('Hình P-02: Ảnh hưởng của Temperature đến Melody sinh ra\n'
             'Temperature thấp = deterministifc, cao = sáng tạo hơn', fontsize=12, fontweight='bold')

for ax, temp in zip(axes, temps):
    notes = model.generate(primer, max_len=3+20, temperature=temp)[0, 3:].cpu().numpy()
    notes = notes[notes < 128]
    ax.step(range(len(notes)), notes, where='post', lw=2, color='mediumpurple')
    ax.fill_between(range(len(notes)), notes, step='post', alpha=0.2, color='mediumpurple')
    ax.set_ylabel(f'T={temp}\npitch', fontsize=9)
    ax.set_ylim(45, 90)
    std_val = np.std(notes) if len(notes) > 1 else 0
    ax.text(0.98, 0.85, f'std={std_val:.1f}', transform=ax.transAxes,
            ha='right', fontsize=9, color='gray')
axes[-1].set_xlabel('Vị trí nốt')
plt.tight_layout()
plt.savefig(SAVE + 'prior_p02_temperature_effect.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved prior_p02_temperature_effect.png')


## Ảnh riêng: Piano Roll nhiều mẫu sinh ra

In [ ]:
model.eval()
n_samples = 6
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle('Hình P-03: Piano Roll của 6 melody sinh ra từ cùng primer (C-E-G)\n'
             'Mỗi lần sinh ra một melody khác nhau (temperature=0.9)', fontsize=12, fontweight='bold')

for i, ax in enumerate(axes.flat):
    primer = torch.tensor([[60, 64, 67]], dtype=torch.long).to(device)
    notes = model.generate(primer, max_len=3+16, temperature=0.9)[0, 3:].cpu().numpy()
    notes = notes[notes < 128]
    rw = max(1, (12*35) // max(len(notes), 1))
    for j, n in enumerate(notes[:16]):
        bar_h = max(2, int((n - 48) / 36 * 25))
        ratio = (n - 48) / 36
        c = plt.cm.RdYlGn(ratio)
        ax.barh(n, 0.85, left=j, height=0.8, color=c, edgecolor='none')
    ax.set_title(f'Mẫu {i+1}', fontsize=10)
    ax.set_xlabel('Thời gian (nốt)'); ax.set_ylabel('Pitch')
    ax.set_ylim(48, 84); ax.set_xlim(0, 16)

plt.tight_layout()
plt.savefig(SAVE + 'prior_p03_piano_roll_samples.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved prior_p03_piano_roll_samples.png')


In [ ]:
torch.save({
    'model_state':model.state_dict(),
    'config':{'vocab_size':130,'d_model':128,'nhead':4,'num_layers':4,'max_len':512},
    'metrics':{'val_loss':val_losses[-1],'perplexity':perplexities[-1],'random_baseline':130},
    'train_losses':train_losses,'val_losses':val_losses,'perplexities':perplexities
},'music_prior.pt')
print(f'Saved: music_prior.pt | Val Loss: {val_losses[-1]:.4f} | Perplexity: {perplexities[-1]:.2f}')
